# Vexar Fleet Intelligence - Notebook 02: Exploratory Data Analysis & Feature Engineering

**Phase**: Stage 2 (Data Engineering + EDA)  
**Author**: Antigravity Data Science & Engineering Team  
**Context**: VexarDrive Technologies Internship Selection Assignment  

---

## Executive Objective
This notebook performs in-depth Exploratory Data Analysis (EDA) on the Vexar Fleet dataset. It evaluates:
1. **Fleet Operating Statistics**: Total distance, driving hours, trip distributions.
2. **Speed Telemetry**: Upper-tail percentiles (P90, P95, P99) and sequential speed change ($\Delta speed$).
3. **3-Axis Accelerometer Baseline & Gravity Deviation**: Empirical baseline median gravity estimation vs. acceleration magnitude deviation from nominal gravity.
4. **Gyroscope Rotational Velocity**: Rotational velocity distribution and upper tail percentiles.
5. **Exposure Normalization & Bias Audit**: Raw extreme counts vs. normalized rates (`events / hour`, `events / 100km`).
6. **Driver vs. Vehicle Anomaly Attribution**: 4-category evidence-based attribution framework (Driver-Linked, Vehicle-Linked, Joint Co-occurrence, Insufficient Evidence).
7. **Candidate Feature Evaluation**: Explicit evaluation table (`KEEP`, `INVESTIGATE`, `REJECT`) with redundancy audit.
8. **Stage 3 Recommendation**: Evidence-based unsupervised modeling strategy recommendation.


In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to Python Path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.ingestion import load_dataset
from src.preprocessing import process_telemetry_features
from src.analysis import analyze_driver_vehicle_relationships, run_full_eda

# Styling Configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Arial'


## Step 1: Load Data & Execute Telemetry Preprocessing

In [ ]:
raw_excel_path = os.path.join(project_root, "data", "raw", "VEXAR_Fleet_Dataset_CANDIDATE_VERSION.xlsx")
proc_dir = os.path.join(project_root, "data", "processed")
fig_dir = os.path.join(project_root, "outputs", "figures")

fleet_data = load_dataset(raw_excel_path)
trip_features = process_telemetry_features(fleet_data)

print(f"Trip Features Computed: {trip_features.shape[0]} trips, {trip_features.shape[1]} metrics.")


## Step 2: Driver-Vehicle Operational Attribution Analysis

In [ ]:
driver_summary, vehicle_summary, usage_matrix, insights = analyze_driver_vehicle_relationships(fleet_data, trip_features, output_dir=proc_dir)

print("=== Driver-Vehicle Operational Cross-Assignment Insights ===")
for k, v in insights.items():
    print(f"{k:32s} : {v}")


## Step 3: Statistical EDA & Visualization Engine

In [ ]:
stats = run_full_eda(fleet_data, trip_features, output_dir=proc_dir, figures_dir=fig_dir)

print("=== Fleet High-Level Statistics ===")
print(f"Total Fleet Distance     : {stats['total_fleet_distance_km']:.2f} KM")
print(f"Total Fleet Driving Time : {stats['total_fleet_driving_hours']:.2f} Hours")
print(f"Empirical Baseline Gravity: {stats['baseline_gravity_median']:.4f} g")
print(f"Speed P90 Threshold      : {stats['speed_stats']['p90']:.2f} km/h")
print(f"Speed P95 Threshold      : {stats['speed_stats']['p95']:.2f} km/h")
print(f"Speed P99 Threshold      : {stats['speed_stats']['p99']:.2f} km/h")


## Step 4: Candidate Feature Evaluation Table

In [ ]:
feature_eval = pd.read_csv(os.path.join(proc_dir, "candidate_features_evaluation.csv"))
feature_eval


## Step 5: Recommended Stage 3 Modeling Strategy

Based on empirical data findings:
- **Dataset Size**: 450 trips across 30 drivers and 30 vehicles over a 1-week window.
- **Ground Truth**: Zero labels for accidents, risk scores, or mechanical faults.
- **Distribution Profile**: Telemetry sensor metrics exhibit heavy right-skewed tails.

### Recommended Modeling Architecture for Stage 3:
**Hybrid Unsupervised Architecture**:
1. **Interpretable Percentile-Based Score**: Exposure-normalized metric rates ($	ext{events / hour}$, $	ext{events / 100km}$) mapped via robust percentile baselines.
2. **Robust Statistical Anomaly Signal**: Isolation Forest / MAD (Median Absolute Deviation) outlier score for multi-dimensional telemetry anomaly detection.
3. **Driver-vs-Vehicle Attribution**: Contextual evidence framing every flagged anomaly as either a **Driver Coaching Candidate** or a **Vehicle Inspection Signal**.
